In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Python is working!")

Python is working!


In [12]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\Lavanya Narang\OneDrive\Documents\GitHub\repoMl\notebooks
['.gitkeep', 'eda.ipynb']


In [13]:
import pandas as pd

s1 = pd.read_csv("../data/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("../data/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("../data/train/train_source3.tsv", sep="\t")
gt = pd.read_csv("../data/train/train_ground_truth.tsv", sep="\t")

print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)
print("Ground Truth:", gt.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [14]:
print("Source 1 columns:")
print(s1.columns.tolist())

print("\nSource 2 columns:")
print(s2.columns.tolist())

print("\nSource 3 columns:")
print(s3.columns.tolist())

print("\nGround Truth columns:")
print(gt.columns.tolist())

Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 2 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 3 columns:
['entity_id', 'business_name', 'business_address', 'country']

Ground Truth columns:
['source1_entity_id', 'matched_entity_ids']


In [15]:
country_counts = pd.concat([
    s1["country"].value_counts().rename("Source 1"),
    s2["country"].value_counts().rename("Source 2"),
    s3["country"].value_counts().rename("Source 3")
], axis=1).fillna(0).astype(int)

country_counts

,Source 1,Source 2,Source 3
country,,,
US,1323633,3016817,3170056
India,883188,2017799,2115547


In [16]:
gt["matched_entity_ids"] = gt["matched_entity_ids"].fillna("").astype(str)

singleton_count = (
    gt["matched_entity_ids"].str.strip() == ""
).sum()

print("Singleton / No-match Source 1 entities:", singleton_count)
print("Total Source 1 entities:", len(gt))
print("Percentage:", round(singleton_count / len(gt) * 100, 2), "%")

Singleton / No-match Source 1 entities: 123247
Total Source 1 entities: 2206821
Percentage: 5.58 %


In [18]:
gt["matched_entity_ids"] = gt["matched_entity_ids"].fillna("").astype(str)

singleton_count = (
    gt["matched_entity_ids"].str.strip() == ""
).sum()

print("Singleton / No-match Source 1 entities:", singleton_count)
print("Total Source 1 entities:", len(gt))
print("Percentage:", round(singleton_count / len(gt) * 100, 2), "%")

Singleton / No-match Source 1 entities: 123247
Total Source 1 entities: 2206821
Percentage: 5.58 %


In [19]:
def count_matches(x):
    x = str(x).strip()
    
    if x == "":
        return 0
    
    return len([i for i in x.split(",") if i.strip()])

gt["match_count"] = gt["matched_entity_ids"].apply(count_matches)

distribution = pd.Series({
    "0 matches": (gt["match_count"] == 0).sum(),
    "1 match": (gt["match_count"] == 1).sum(),
    "2 matches": (gt["match_count"] == 2).sum(),
    "3+ matches": (gt["match_count"] >= 3).sum()
})

distribution

0 matches      123247
1 match        119157
2 matches      375212
3+ matches    1589205
dtype: int64

In [20]:
from collections import defaultdict

reverse_matches = defaultdict(list)

for _, row in gt.iterrows():
    matches = str(row["matched_entity_ids"]).strip()
    
    if matches == "":
        continue
    
    for entity_id in matches.split(","):
        entity_id = entity_id.strip()
        
        if entity_id:
            reverse_matches[entity_id].append(row["source1_entity_id"])

multiple_matches = {
    entity_id: s1_ids
    for entity_id, s1_ids in reverse_matches.items()
    if len(s1_ids) > 1
}

print("S2/S3 IDs matched to multiple Source 1 entities:", len(multiple_matches))

S2/S3 IDs matched to multiple Source 1 entities: 0


In [21]:
all_sources = pd.concat([
    s1.assign(source="S1"),
    s2.assign(source="S2"),
    s3.assign(source="S3")
], ignore_index=True).set_index("entity_id")

examples = []

for _, row in gt.iterrows():
    matches = str(row["matched_entity_ids"]).strip()

    if matches == "":
        continue

    source1_id = row["source1_entity_id"]

    for match_id in matches.split(","):
        match_id = match_id.strip()

        if match_id in all_sources.index:
            examples.append({
                "source1_id": source1_id,
                "source1_name": all_sources.loc[source1_id, "business_name"],
                "source1_address": all_sources.loc[source1_id, "business_address"],
                "matched_id": match_id,
                "matched_name": all_sources.loc[match_id, "business_name"],
                "matched_address": all_sources.loc[match_id, "business_address"]
            })

        if len(examples) >= 10:
            break

    if len(examples) >= 10:
        break

examples_df = pd.DataFrame(examples)

examples_df

,source1_id,source1_name,source1_address,matched_id,matched_name,matched_address
0,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S2-681193310,Maure Wilblims Colombier Inc,NaN
1,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S2-743505751,Maure Williams Colombier,NaN
2,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York"
3,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York"
4,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-860443364,Maure Williams Inc Center,NaN
5,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...",S2-249013014,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, ..."
6,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...",S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, ..."
7,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...",S3-478195123,Raj Investments எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, ..."
8,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...",S3-384364074,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, ..."
9,S1-343815751,Dahlia Power Reliable Scientific LLC,"630 45th Terrace, Kansas City, MO",S2-790675320,Dahlia Power Reliable,"KANSAS CITY, MO, 630 45ND TERRACE, null"


In [22]:
import re

def extract_postcode(address):
    if pd.isna(address):
        return ""
    
    address = str(address)
    matches = re.findall(r"\b\d{5,6}\b", address)
    
    return matches[-1] if matches else ""

for df in [s1, s2, s3]:
    df["postcode"] = df["business_address"].apply(extract_postcode)

postcode_coverage = pd.DataFrame({
    "Source 1": [
        (s1["postcode"] != "").sum(),
        round((s1["postcode"] != "").mean() * 100, 2)
    ],
    "Source 2": [
        (s2["postcode"] != "").sum(),
        round((s2["postcode"] != "").mean() * 100, 2)
    ],
    "Source 3": [
        (s3["postcode"] != "").sum(),
        round((s3["postcode"] != "").mean() * 100, 2)
    ]
}, index=["Records with postcode", "Coverage (%)"])

postcode_coverage

,Source 1,Source 2,Source 3
Records with postcode,147257.00,369140.00,385727.0
Coverage (%),6.67,7.33,7.3


In [23]:
import re

def extract_postcode(address):
    if pd.isna(address):
        return ""
    
    address = str(address)
    matches = re.findall(r"\b\d{5,6}\b", address)
    
    return matches[-1] if matches else ""

for df in [s1, s2, s3]:
    df["postcode"] = df["business_address"].apply(extract_postcode)

postcode_coverage = pd.DataFrame({
    "Source 1": [
        (s1["postcode"] != "").sum(),
        round((s1["postcode"] != "").mean() * 100, 2)
    ],
    "Source 2": [
        (s2["postcode"] != "").sum(),
        round((s2["postcode"] != "").mean() * 100, 2)
    ],
    "Source 3": [
        (s3["postcode"] != "").sum(),
        round((s3["postcode"] != "").mean() * 100, 2)
    ]
}, index=["Records with postcode", "Coverage (%)"])

postcode_coverage

,Source 1,Source 2,Source 3
Records with postcode,147257.00,369140.00,385727.0
Coverage (%),6.67,7.33,7.3


In [25]:
import sys
sys.path.append("..")

from src.split import make_splits

In [26]:
splits = make_splits(
    gt,
    s1,
    val_fraction=0.2,
    seed=42
)

print("Train:", len(splits["train_source1_ids"]))
print("Validation:", len(splits["val_source1_ids"]))

Train: 1765456
Validation: 441365
